In [ ]:
import os
import sys

import collections
import xarray as xr
xr.set_options(keep_attrs=True)
import netCDF4 as nc
import numpy as np
np.seterr(divide='ignore', invalid='ignore')
import metpy.calc as mp
from metpy.units import units
from scipy.stats import ttest_ind, ttest_rel
from datetime import datetime

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from shapely.geometry.polygon import LinearRing

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
from matplotlib import cm
from matplotlib.colors import ListedColormap,LinearSegmentedColormap
import cmocean.cm as cmo

# settings
%config InlineBackend.figure_format = 'retina'

In [ ]:
def windSpd(u,v):
   windSpd=np.sqrt(u**2 + v**2)
   return windSpd

def froude(N,h,Uspd):
   fr=(N*h)/Uspd
   return fr

def froude_from_upstream_vars(topo,u,v,ts,ta,z,
                              p2wind=800,
                              p2n=600,
                              oro_lat_bounds=None,
                              oro_lon_bounds=None,
                              ups_lat_bounds=None,
                              ups_lon_bounds=None):
    ## Clip topo data to corresponding lat/lon bounds
    print('Clipping topography...')
    tymin=oro_lat_bounds[0] ; tymax=oro_lat_bounds[1]
    txmin=oro_lon_bounds[0] ; txmax=oro_lon_bounds[1]
    h=topo.sel(lat=slice(tymin,tymax), lon=slice(txmin,txmax))
    
    ## Define pressure level thresholds for wind and N vars
    surface_plev=1000
    wind_p2lev=p2wind
    wind_p2idx=int(np.where(u.lev==wind_p2lev)[0])
    n_p2lev=p2n
    n_p2idx=int(np.where(u.lev==n_p2lev)[0])

    ## Clip upstream climate data and calculate zonal regional means
    print('Calculating Zonal Mean Upstream Uspd...')
    ymin=ups_lat_bounds[0] ; ymax=ups_lat_bounds[1]
    xmin=ups_lon_bounds[0] ; xmax=ups_lon_bounds[1]
    # Wind Speed
    uwnd=u[:,0:wind_p2idx,:,:].sel(lat=slice(ymin,ymax), lon=slice(xmin,xmax))
    vwnd=v[:,0:wind_p2idx,:,:].sel(lat=slice(ymin,ymax), lon=slice(xmin,xmax))
    Uspd=windSpd(uwnd,vwnd).mean(dim=['lev','lon'])
    # Buoyancy Frequency
    print('Calculating Zonal Mean Upstream N...')
    t0=ts.sel(lat=slice(ymin,ymax), lon=slice(xmin,xmax))
    t1=ta[:,n_p2idx,:,:].sel(lat=slice(ymin,ymax), lon=slice(xmin,xmax))
    z0=topo.sel(lat=slice(ymin,ymax), lon=slice(xmin,xmax)) # get surface height from etopo dataset
    z1=z[:,n_p2idx,:,:].sel(lat=slice(ymin,ymax), lon=slice(xmin,xmax)) # height of atm at plevel
    g=np.full((len(t0.lat),len(t0.lon)), 9.81) # gravity
    theta=t1*((surface_plev/p1)**(2/7))
    dtdz=(t1-t0)/(z1-z0)
    dthetadz=((surface_plev/p1)**(2/7)) * (.00976 - np.abs(dtdz))
    N=np.sqrt((g/theta)*dthetadz).mean(dim='lon',skipna=True)
    
    ## Calculate Froude
    print('Calculating Froude...')
    Fr=froude(N,h,Uspd)
    print('Done.')
    return(Fr)

In [ ]:
### +++ LOAD DATA +++ ###

# MERRA-2 reanalysis
merra2 = { 'TS':{}, 'T':{}, 'H':{}, 'U':{}, 'V':{} }
for varn in merra2.keys():
    merra2[varn] = xr.open_dataset(f'merra2.{varn}.1980-2022.monthly.nc')[varn].groupby("time.month").mean(dim='time')

# topography data
dpath0='/Users/dervlamk/OneDrive/research/topo_biases/topo_files' #'../topo_files' # top level data directory
# store file paths
files={ 'etopo':{}, 'ctrl':{}, 'max':{}, 'p85':{}, 'p80':{}, '1.5env':{}, '1env':{} }
files['etopo'] = f'{dpath0}/obs.etopo5.zsurf.nc' #obs.etopo05.zsurf.nc
files['ocn_mask'] = f'{dpath0}/Z2HX2fromZ1QX1N.BS1.nc' #e2pt1.ctrl.zsurf.nc'
files['ctrl'] = f'{dpath0}/Z2HX2fromZ1QX1N.BS1.nc' #e2pt1.ctrl.zsurf.nc'
files['max'] = f'{dpath0}/giss.maxtopo.nc' #e2pt1.max.global.zsurf.nc'
files['p85'] = f'{dpath0}/giss.85topo.cam.sam.nc' #e2pt1.85p.ca-sa.zsurf.nc'
files['p80'] = f'{dpath0}/giss.80topo.cam.sam.nc' #e2pt1.80p.ca-sa.zsurf.nc'
files['1.5env'] = f'{dpath0}/giss.1.5s-env.topo.cam.sam.nc' #e2pt1.1pt5sEnv.ca-sa.zsurf.nc'
files['1env'] = f'{dpath0}/giss.1s-env.topo.cam.sam.nc' #e2pt1.1sEnv.ca-sa.zsurf.nc'

# read in data
topo={ 'etopo':{}, 'ctrl':{}, 'max':{}, 'p85':{}, 'p80':{}, '1.5env':{}, '1env':{} }
for key in files.keys():
    # need to process the etopo data first so coordinate conventions match the model data
    if key in ['etopo']:
        etopo_full = xr.open_dataset(files[key]).ROSE
        # rename dimensions to match model data
        etopo_full = etopo_full.rename({'ETOPO05_Y':'lat', 'ETOPO05_X':'lon'}) 
        # switch from 0:360 to -180:180 lons
        nx = len(etopo_full.lon)
        lons = np.linspace(-180,180,nx)
        etopo_full['lon']=lons
        etopo_full=etopo_full.roll(lon=2160)
        topo[key] = etopo_full.where(etopo_full>0,np.nan) # remove bathymetry
    else:
        # interpolate model data to high resolution
        topo[key] = xr.open_dataset(files[key]).zatmo.interp(lat=topo['etopo'].lat, lon=topo['etopo'].lon)
        
# load ocean mask
ocn_mask = xr.open_dataset(files['ocn_mask']).focean

In [ ]:
#=== CALCULATE FROUDE

# each dict defines the bounds of the orographic and upstream regions
# relevant for calculating froude at those latitudes informed by the
# dominant wind regimes
regions = [
    dict(oro_lat=[-55, -30], oro_lon=[-80, -60],
         ups_lat=[-55, -30], ups_lon=[-95, -90]),
    dict(oro_lat=[-29.99, -10], oro_lon=[-80, -60],
         ups_lat=[-29.99, -10], ups_lon=[-55, -50]),
    dict(oro_lat=[-9.99, 11], oro_lon=[-90, -69],
         ups_lat=[-9.99, 11], ups_lon=[-65, -60]),
    dict(oro_lat=[11.01, 25], oro_lon=[-110, -80],
         ups_lat=[11.01, 25], ups_lon=[-75, -70])
]

froude = { 'etopo':{}, 'ctrl':{}, 'max':{}, 'p85':{}, 'p80':{}, '1.5env':{}, '1env':{} }
results = []
# set info about what upper levels to average over for winds and N
p2wind=800
p2n=600
for key in topo.keys():
    for region in regions:
        # use pre-defined function to calculate froude
        f = froude_from_upstream_vars(
            topo[key], merra2['U'], merra2['V'], merra2['TS'], merra2['T'], merra2['H'],
            p2wind=p2wind, p2n=p2n,
            oro_lat_bounds=region['oro_lat'], oro_lon_bounds=region['oro_lon'],
            ups_lat_bounds=region['ups_lat'], ups_lon_bounds=region['ups_lon']
        )
        results.append(f)
        
    # concat the subregions over latitude and get rid of 0 values
    froude[key] = xr.concat(results, dim="lat", coords="minimal").where(lambda x: x > 0, np.nan)
    froude[key].attrs.update(
        long_name="Froude",
        wind_levels=f"1000 to {p2wind} hPa",
        N_levels=f"1000 to {p2n} hPa",
    )
    froude[key].attrs.pop("units", None)  # safe remove if it exists as fr is dimensionless


In [ ]:
#=== Clean up model froude

# set critical Froude value
frCrit = 1

# original model resolution
target_lats = topo['ctrl'].lat
target_lons = topo['ctrl'].lon

# init new dictionary for storing coarsened froude data for model bcs
e2pt1_froude = { 'ctrl':{}, 'max':{}, 'p85':{}, 'p80':{}, '1.5env':{}, '1env':{} }

for key in e2pt1_froude.keys():
    # regrid e2.1 topo bc froude to original model resolution
    dat = e2pt1_froude[key].interp(lat=target_lats, lon=target_lons)
    # keep only Froude values above critical value (>1) and non-ocean cells
    ones = xr.ones_like(dat)
    e2pt1_froude[key] = dat.where(dat>=frCrit, np.nan).where(ocn_mask>0.25,np.nan)


#=== Clean up obs froude
ones = xr.ones_like(froude['etopo'])
etopo_froude = froude['etopo'].where(froude['etopo']>=frCrit, np.nan)

In [ ]:
# store mean MERRA-2 U and V vectors for plotting
u = merra2['U'].sel(lev=850).mean(dim='month')
v = merra2['U'].sel(lev=850).mean(dim='month')

In [ ]:
# -------------------- #
#       SETTINGS       #
# -------------------- #
text_kw={'color':'k', 'weight':'bold', 'size':'x-large', 'horizontalalignment':'left'}
arr_text_kw={'color':'blue', 'weight':'bold', 'size':'medium', 'horizontalalignment':'left', 'verticalalignment':'bottom'}
arrow_kw={'color':'blue', 'linewidth':.01, 'shrink':0.0}
# cmap specs
tcmap=cmo.deep
newcolors=tcmap(np.linspace(0.1, 1, 200))
white=np.repeat(np.array([[1, 1, 1, 1],[1, 1, 1, 1]]),25, axis=0)
cmap=ListedColormap(np.vstack((white, newcolors)))
vmin=0
vmax=5
# vector specs
skip_n=5
w=0.003
scalef=35
key_length=5
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-110,-60,-55,25]

# -------------------- #
#          FIG         #
# -------------------- #
fig, ax = plt.subplots(nrows=1, ncols=21, figsize=(25,5),
                       layout='constrained',
                       subplot_kw={'projection':proj},
                       gridspec_kw={'width_ratios': [3, 0.5, 0.15, 3, 0.5, 0.15, 3, 0.5, 0.15, 3, 0.5, 0.15, 3, 0.5, 0.15, 3, 0.5, 0.15, 3, 0.5, 0.15]}) #

## ETOPO Froude
ax[0].text(-109.5,26,'a. ETOPO05',**text_kw,zorder=100)
cf=ax[0].pcolormesh(etopo_froude.lon, etopo_froude.lat, etopo_froude.mean(dim='month'),
                    cmap=cmap, vmin=vmin, vmax=vmax, transform=trans)
q1=ax[0].quiver(u.lon[::skip_n], u.lat[::skip_n],
                u[::skip_n,::skip_n], v[::skip_n,::skip_n],
                color='k', width=w, scale=scalef, scale_units='inches', units='height', transform=trans, zorder=10)
qk=ax[0].quiverkey(q1, .9, 1.025, key_length, rf'{key_length} m/s', labelcolor='k', labelpos='W')
ax[0].annotate('TE', xy=(-94.25,18), xytext=(-95.25,23), label='_Hidden', arrowprops=arrow_kw, **arr_text_kw, zorder=100)
ax[0].annotate('PG', xy=(-84,12.2), xytext=(-80,15.5), label='_Hidden', arrowprops=arrow_kw, **arr_text_kw, zorder=100)
ax[0].annotate('PN', xy=(-79,9.5), xytext=(-76,13.25), label='_Hidden', arrowprops=arrow_kw, **arr_text_kw, zorder=100)
ax[0].set(aspect='auto')
ax[0].coastlines(color='k')
ax[0].add_feature(cfeature.OCEAN, fc=(0.9, 0.9, 0.9), zorder=0)
ax[0].add_feature(cfeature.LAND, fc=(0.9, 0.9, 0.9), zorder=0)
ax[0].set_extent(map_bnds, crs=trans)
gl=ax[0].gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
gl.xlocator=mticker.FixedLocator([-100,-90,-80,-70])
gl.bottom_labels=True; gl.left_labels=True; gl.top_labels=False; gl.right_labels=False
gl.xformatter=LONGITUDE_FORMATTER
gl.yformatter=LATITUDE_FORMATTER
gl.xlabel_style={'color': 'black', 'weight': 'normal'}
gl.ylabel_style={'color': 'black', 'weight': 'normal'}
# heatmap
temp=frObs.mean(dim='month', skipna=True).interp_like(frGiss_lowres).max(dim='lon')
vec=np.flip(np.array([temp]))
frObs_zm=vec.reshape(vec.shape[1], 1)
sns.heatmap(frObs_zm, vmin=vmin, vmax=vmax, cmap=cmap, yticklabels=False, xticklabels=False, square=True, cbar=False, ax=ax[1])
ax[1].set(aspect=0.55)


# these are extra axes just for aesthetic spacing
ax[2].set_visible(False)
ax[5].set_visible(False)
ax[8].set_visible(False)
ax[11].set_visible(False)
ax[14].set_visible(False)
ax[17].set_visible(False)
ax[20].set_visible(False)


## E2.1 Topo BCs Froude
for i,key in enumerate(e2pt1_froude.keys()):
    axn=[3,6,9,12,15,18]
    n=axn[i]
    letter=['b. ','c. ','d. ','e. ','f. ','g. ', 'h. ', 'i. ']
    ax[n].text(-109.5,26,letter[i]+key,**text_kw,zorder=100)
    
    ## plot Froude map
    ax[n].pcolormesh(e2pt1_froude[key].lon, e2pt1_froude[key].lat, e2pt1_froude[key].mean(dim='month'),
                     cmap=cmap, norm=norm, transform=trans)
    ax[n].set(aspect='auto')
    ax[n].coastlines(color='k')
    ax[n].add_feature(cfeature.OCEAN, fc=(0.9, 0.9, 0.9), zorder=0)
    ax[n].add_feature(cfeature.LAND, fc=(0.9, 0.9, 0.9), zorder=0)
    ax[n].set_extent(map_bnds, crs=trans)
    gl=ax[n].gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.xlocator=mticker.FixedLocator([-100,-90,-80,-70])
    gl.bottom_labels=True; gl.left_labels=False; gl.top_labels=False; gl.right_labels=False
    gl.xformatter=LONGITUDE_FORMATTER
    gl.yformatter=LATITUDE_FORMATTER
    gl.xlabel_style={'color': 'black', 'weight': 'normal'}
    gl.ylabel_style={'color': 'black', 'weight': 'normal'}
    
    ## plot max Froude heat map
    vec=np.flip(np.array([frGiss_lowres[i].mean(dim='month').max(dim='lon')]))
    frGiss_zm=vec.reshape(vec.shape[1], 1)
    sns.heatmap(frGiss_zm, vmin=vmin, vmax=vmax, cmap=cmap, yticklabels=False, xticklabels=False, square=True, cbar=False, ax=ax[n+1])
    ax[n+1].set(aspect=.55)

    
# add colorbar
levels=np.linspace(vmin, vmax, 11)
norm=mpl.colors.BoundaryNorm(levels, cmap.N)
cf=mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
cax=fig.add_axes([.245, -0.075, 0.5, 0.05])
cbar=fig.colorbar(cf, orientation='horizontal', extend='max', cax=cax)
cbar.set_label('$F$', x=-0.025, labelpad=-32, rotation=0, size='xx-large', fontweight='bold', ha='center')
cbar.ax.tick_params(labelsize='x-large')
for tick in cbar.ax.yaxis.get_major_ticks():
    tick.label2.set_fontweight('normal')
